In [3]:
import pandas as pd
from typing import Optional, Dict
import sys
from pathlib import Path
from datetime import datetime
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.filtering import apply_filters, only_korean, has_value, FilterValue

In [22]:
# Helper: 
# 
def pick_first_existing_column(df, candidates, *, required=True):
    """
    candidates 중 df에 실제로 존재하는 첫 번째 컬럼 이름을 반환.
    """
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of the candidate columns exist: {candidates}")
    return None

#  
def generate_sentence_df_one_filters(
    df: pd.DataFrame,
    df_sen: pd.DataFrame,
    filters: Optional[Dict[str, FilterValue]] = None,
    window: int = 0,
    process_fn=None
) -> pd.DataFrame:
    """
    단일 DF에서 filters(dict)로 필터링한 결과를 문장과 합쳐서 출력.
    window>0이면 word_id2 주변으로 확장.
    """
    # 1) 필터링
    target_df = apply_filters(df, filters)
    print(f"필터링 완료: {len(target_df)}행.")

    # 2) window 확장(옵션)
    if window > 0:
        df_use = expand_window(target_df, df, window)
    else:
        df_use = target_df
    print(f"window 적용 완료: {len(df_use)}행.")

    # 3) sentence form 컬럼 자동 선택
    sen_form_col = pick_first_existing_column(
        df_sen,
        ["sentence_form", "s_form", "form", "word_form"]
    )
    
    # 4) 문장 텍스트 결합

    sen_use = (
        df_sen.rename(columns={sen_form_col: "sentence_form"})
    )
    
    result_df = sen_use.merge(df_use, on="sen_id", how="right")
    print("문장 결합 완료.")

    # 4) 후처리
    if process_fn:
        result_df = process_fn(result_df)
        print("후처리 완료.")
    
    return result_df

###오류 후보 행들의 sen_id와 word_id를 기준으로 주변 window만큼 확장한 행을 추출.    #return result_df
def expand_window(taget_df, df, window=2):
    """
    오류 후보 행들의 sen_id와 word_id를 기준으로 주변 window만큼 확장한 행을 추출.
    """
    # 오류 후보들만 뽑고
    taget_row = taget_df[['sen_id', 'word_id']].dropna().copy()
    taget_row['word_id'] = taget_row['word_id'].astype(int)

    # window 범위로 확장
    expanded_rows = []
    for _, row in taget_row.iterrows():
        sid = row['sen_id']
        wid = row['word_id']
        for offset in range(-window, window + 1):
            expanded_rows.append((sid, wid + offset))

    # 중복 제거
    expanded_df = pd.DataFrame(expanded_rows, columns=['sen_id', 'word_id']).drop_duplicates()

    # 기준 df와 merge
    df['word_id'] = df['word_id'].astype('Int64')  # Null 허용 정수로
    result_df = pd.merge(df, expanded_df, on=['sen_id', 'word_id'], how='inner')
    result_df = result_df.sort_values(by=['sen_id', 'word_id']).reset_index(drop=True)

    return result_df


In [ ]:
#파일 읽어오기
START_TIME = datetime.now()
CSV_PATH = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv"
df_word = pd.read_csv(CSV_PATH)
print(f"파일 읽기 완료 - {datetime.now().strftime('%Y-%m-%d_%H:%M')}")

파일 읽기 완료 - 20260405_09:11:09


In [7]:
df = df_word.copy()
# ---------------------------------------------
# f_EN_no, f_EN_no_sub, f_EN_label 생성, V0_form, V0_label, V0_No 생성
# ---------------------------------------------
START = datetime.now()
print(f"시작 - {START.strftime("%Y%m%d_%H-%M-%S")}")

# ---------------------------------------------
# 1. f_en_no, f_en_no_sub, f_en_label 생성
# ---------------------------------------------
if True: 
    df = df.drop(columns=["f_EN_form", "f_EN_label", "f_EN_No"], errors="ignore") # 기존에 있으면 제거 (중복 방지)

    # -1이면 자기 word_id2로 대체
    df["f_word_id"] = df["f_word_id"].where(
        df["f_word_id"] != -1,
        df["word_id"]
    )

    lookup = (
        df[
            [
                "sen_id",
                "word_id",
                "EN_form",
                "EN_No",
                "EN_No_sub",
                "EN_label",
                "EP_form"
            ]
        ]
        .rename(
            columns={
                "word_id": "f_word_id",
                "EN_form": "f_EN_form",
                "EN_No": "f_EN_No",
                "EN_No_sub": "f_EN_No_sub",
                "EN_label": "f_EN_label",
                "EP_form": "f_EP_form"
            }
        )
    )

    # ------------------------------------------
    # sen_id + f_word_id 기준 self merge
    # ------------------------------------------
    df = df.merge(
        lookup,
        on=["sen_id", "f_word_id"],
        how="left"
    )

print(f"f_EN 생성 완료 - {datetime.now()-START}")
# ---------------------------------------------
# 2. V0_form, V0_label, V0_No 생성
# ---------------------------------------------
START = datetime.now()
print(f"시작 - {START.strftime("%Y%m%d_%H-%M-%S")}")

if True:
    df = df.drop(columns=["V0_form", "V0_label", "V0_No"], errors="ignore") # 기존에 있으면 제거 (중복 방지)

    df["V0_word_id"] = df["V0_word_id"].where(
        df["V0_word_id"] != -1,
        df["word_id"]
    )

    lookup_V = (
        df[
            [
                "sen_id",
                "word_id",
                "V_form",
                "V_label",
                "V_No"
            ]
        ]
        .rename(
            columns={
                "word_id": "V0_word_id",
                "V_form": "V0_form",
                "V_label": "V0_label",
                "V_No": "V0_No"
            }
        )
    )

    df = df.merge(
        lookup_V,
        on=["sen_id", "V0_word_id"],
        how="left"
    )

print(f"V0 생성 완료 - {datetime.now()-START}")
df.columns

시작 - 20260405_09-11-54
f_EN 생성 완료 - 0:00:16.350218
시작 - 20260405_09-12-10
V0 생성 완료 - 0:00:26.472219


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub',
       'f_EN_label', 'f_EP_form', 'V0_form', 'V0_label', 'V0_No'],
      dtype='object')

In [ ]:
SEN_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen.csv"     
df_sen = pd.read_csv(SEN_CSV, low_memory=False)
#document 파일 읽기
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보_ver.1.2.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)
#file 파일 읽기
FILE_CSV = r"..\csv\original_csv\세종문어_file_정보_ver.1.1.csv"
df_file = pd.read_csv(FILE_CSV, low_memory=False)

print(f"파일 읽기 완료 - {datetime.now().strftime('%Y%m%d_%H:%M:%S')}")
print(f"파일 읽기 완료 - {datetime.now().strftime('%Y%m%d_%H:%M:%S')}")
print(f"df_file: {df_file.columns.tolist()}")
print(f"df_docu: {df_docu.columns.tolist()}")
print(f"df_sen: {df_sen.columns.tolist()}")
print(f"df_file: {len(df_file):,}행, df_docu: {len(df_docu):,}행, df_sen: {len(df_sen):,}행")

파일 읽기 완료 - 20260405_09:12:47
파일 읽기 완료 - 20260405_09-12-47
df_file: ['file_id', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4', 'sent_count', 'head_count', 'body_count', 'body_has_E_count', 'body_not_quote_count', 'body_not_quote_and_었_count', '었_결합_오즈비', '었_결합_로그오즈비', '었_결합_등급', '었_결합_성향']
df_docu: ['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4', 'docu_sen_count', 'docu_sen_count_has_E', 'docu_sen_count_not_quote', 'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote', 'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_count', 'docu_dominant_ratio', 'docu_sent_count', 'docu_head_count', 'docu_body_count', 'docu_body_has_E_count', 'docu_body_not_quote_count', 'docu_body_not_quote_and_었_count', 'docu_었_결합_오즈비', 'docu_었_결합_로그오즈비', 'docu_었_결합_등급', 'docu_었_결합_성향', 'category2', 'file_sent_count', 'file_head_count', 'file_body_cou

In [ ]:
df = df.merge(
    df_docu[['docu_id', 
             'category', 'category2','매체', '파일제목', '저자', '출판사', '출판연도', 'head', 
             'docu_sen_count', 'docu_sen_count_has_E', 'docu_sen_count_not_quote', 'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote', 'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_count' ]],
    on="docu_id",
    how="left",
    validate="many_to_one"
)
print(f"df에 document 정보 merge 완료 - {datetime.now().strftime('%Y%m%d_%H:%M:%S')}")

df.columns

df에 document 정보 merge 완료 - 20260405_09-12-47


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub',
       'f_EN_label', 'f_EP_form', 'V0_form', 'V0_label', 'V0_No', 'category',
       'category2', '매체', '파일제목', '저자', '출판사', '출판연도', 'head',
       'docu_sen_count', 'docu_sen_count_has_E', 'docu_sen_count_not_quote',
       'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote',
       'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_count'],
      dtype='object')

In [ ]:
#document２ 파일 읽기
DOCU_CSV = r"..\csv\original_csv\세종문어_신문_document_었_결합_분류.csv"
df_docu2 = pd.read_csv(DOCU_CSV, low_memory=False)

# ---
# document 정보에서 필요한 컬럼 병합
# ---
def safe_merge(df, df_add, key, cols):
    cols_to_use = [key] + [c for c in cols if c not in df.columns]
    return df.merge(df_add[cols_to_use], on=key, how="left")

df = safe_merge(df, df_docu2, "docu_id", [' 었_결합_분류 '])

df["분석대상"] = df[" 었_결합_분류 "].notna().map({True: "대상", False: "제외"})
# 컬럼명 변경
df = df.rename(columns={
    " 었_결합_분류 ": "었_결합_분류",
})
df['었_결합_성향'] = df['었_결합_분류'].map({
    1: '회피',
    2: '회피',
    3: '보통',
    4: '선호',
    5: '선호'
})

print(f"df에 document 정보 merge 완료 - {datetime.now().strftime('%Y%m%d_%H:%M:%S')}")
df.columns

df에 document 정보 merge 완료 - 20260315_18-25-13


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id2', 'rev_word_id2',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form_0', 'V_label_0',
       'V_form', 'V_label', 'EP_form', 'EP_label', 'EN_form', 'EN_label',
       'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end', 'sent_end_V',
       'sent_end_V_in_quote', 'quote_type', 'vx_len', 'Next_VX_No',
       'Next_VX_form', 'vx0_No', 'vx0_form', 'vx0_order', 'V_word_id',
       'f_word_id', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label',
       'f_EP_form', 'V0_form', 'V0_label', 'V0_No', 'category', '매체', '파일제목',
       '저자', '출판사', '출판연도', 'head', 'sen_count', 'sen_count_has_E',
       'sen_count_not_quote', 'sen_count_has_E_not_quote',
       'base_count_not_quote', 'dominant_EN_No', 'dominant_EN_label',
       'dominant_count', 'dominant_ratio', '었_결합_분류', '분석대상', '었_결합_성향'],
      dtype='object')

In [ ]:
# 커럼명 중복 제거(_x, _y)
y_cols = [c for c in df.columns if c.endswith("_y")]
df = df.drop(columns=y_cols)

df.columns = [c[:-2] if c.endswith("_x") else c for c in df.columns]
print(f"df columns after merge - {datetime.now().strftime('%Y%m%d_%H:%M:%S')}")
df.columns

df columns after merge - 20260405_09-12-47


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub',
       'f_EN_label', 'f_EP_form', 'V0_form', 'V0_label', 'V0_No', 'category',
       'category2', '매체', '파일제목', '저자', '출판사', '출판연도', 'head',
       'docu_sen_count', 'docu_sen_count_has_E', 'docu_sen_count_not_quote',
       'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote',
       'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_count'],
      dtype='object')

In [23]:
# --
# 필터 넣어서 확인 문장 출력
# --
from datetime import datetime
OUT_DIR = r"..\csv\문장확인"
PREFIX = "VX_오류"
OUT_GENERATE_TIME = datetime.now()

#오류 확인 필터
FILTER_ERROR_VX = {
    "VX0_No": -1,
    "V_label": "VX"
}

#필터
'''filters: Dict[str, FilterValue] = {
    "category": ["인문사회"],
    "V_label": ["XSV", "XSA"],
    "outcome_total": lambda s: s >= 500,
    "outcome_total": lambda s: (s >= 20) & (~s.isin(list([4999, 2999]))),
    "V_form": only_korean, #한글로만 이루어져있나
    "V_label": lambda s: ~has_value(s), #값이 들어있나
    }'''

filters =  {
    'sent_end_V': True,
    '매체': '신문',
    '분석대상': '대상',    
    }
'''
only_korean 함수는 아래와 같은 효과임. 
filters = {
    "V_form": lambda s: s.astype(str).str.fullmatch(r"[가-힣ㄱ-ㅎㅏ-ㅣ]+", na=False)
}'''

#------------------------------
#df_sen에 필요한 정보만 남김
df_sen = df_sen.merge(df_docu[['docu_id', '매체', '파일제목','출판사', 'head']], on='docu_id', how='left')
keep_cols = ['sen_id', 'file_id', 'docu_id', 'sen_num',
       'sentence_form', 'speaker', 'sen_len', '매체', '파일제목', '출판사', 'head'
]
keep_cols = [c for c in keep_cols if c in df_sen.columns]
df_sen = df_sen[keep_cols].copy()
#--------------------------------
out = generate_sentence_df_one_filters(df, df_sen, filters=FILTER_ERROR_VX, window=2)
print(f"out columns after generation  - 시작: {OUT_GENERATE_TIME.strftime('%Y%m%d_%H:%M:%S')}, 종료: {datetime.now().strftime('%Y%m%d_%H:%M:%S')}")

out.columns

필터링 완료: 16082행.
window 적용 완료: 61605행.
문장 결합 완료.
out columns after generation  - 시작: 20260405_09:37:45, 종료: 20260405_09:37:50


Index(['sen_id', 'file_id_x', 'docu_id_x', 'sen_num', 'sentence_form',
       'speaker', 'sen_len', '매체_x', '파일제목_x', '출판사_x', 'head_x', 'ID',
       'file_id_y', 'docu_id_y', 'word_id', 'rev_word_id', 'form/label',
       'N_form', 'N_label', 'V_No', 'V_form', 'V_label', 'V_form_old',
       'V_label_old', 'EP_form', 'EP_label', 'EN_form', 'EN_label', 'J_form',
       'J_label', 'EN_No', 'EN_No_sub', 'sent_end', 'sent_end_V',
       'sent_end_V_in_quote', 'quote_type', 'VX_len', 'Next_VX_No',
       'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order', 'V0_word_id',
       'f_word_id', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label',
       'f_EP_form', 'V0_form', 'V0_label', 'V0_No', 'category', 'category2',
       '매체_y', '파일제목_y', '저자', '출판사_y', '출판연도', 'head_y', 'docu_sen_count',
       'docu_sen_count_has_E', 'docu_sen_count_not_quote',
       'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote',
       'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_cou

In [18]:
# 컬럼명 중복 제거(_x, _y)
y_cols = [c for c in out.columns if c.endswith("_y")]
out = out.drop(columns=y_cols)

out.columns = [c[:-2] if c.endswith("_x") else c for c in out.columns]

# 컬럼 순서 재배치
front_cols = ['ID','sen_id',"speaker", 'sentence_form']
last_cols = [
              'category2','매체', '파일제목', '저자', '출판사', '출판연도', 'head', 
             'docu_sen_count', 'docu_sen_count_has_E', 'docu_sen_count_not_quote', 'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote', 'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_count' ]
rest_cols = [col for col in out.columns if col not in front_cols + last_cols]
out = out[front_cols + rest_cols + last_cols]

print(f"out columns after generation - 시작: {OUT_GENERATE_TIME.strftime('%Y%m%d_%H:%M:%S')}, 종료: {datetime.now().strftime('%Y%m%d_%H:%M:%S')}")

out.columns

out columns after generation - 시작: 20260405_09:30:20, 종료: 20260405_09:30:24


Index(['ID', 'sen_id', 'speaker', 'sentence_form', 'file_id', 'docu_id',
       'sen_num', 'sen_len', 'word_id', 'rev_word_id', 'form/label', 'N_form',
       'N_label', 'V_No', 'V_form', 'V_label', 'V_form_old', 'V_label_old',
       'EP_form', 'EP_label', 'EN_form', 'EN_label', 'J_form', 'J_label',
       'EN_No', 'EN_No_sub', 'sent_end', 'sent_end_V', 'sent_end_V_in_quote',
       'quote_type', 'VX_len', 'Next_VX_No', 'Next_VX_form', 'VX0_No',
       'VX0_form', 'VX0_order', 'V0_word_id', 'f_word_id', 'f_EN_form',
       'f_EN_No', 'f_EN_No_sub', 'f_EN_label', 'f_EP_form', 'V0_form',
       'V0_label', 'V0_No', 'category', 'category2', '매체', '파일제목', '저자', '출판사',
       '출판연도', 'head', 'docu_sen_count', 'docu_sen_count_has_E',
       'docu_sen_count_not_quote', 'docu_sen_count_has_E_not_quote',
       'docu_base_count_not_quote', 'docu_dominant_EN_No',
       'docu_dominant_EN_label', 'docu_dominant_count'],
      dtype='object')

In [24]:
#저장하기

save_folder = Path(OUT_DIR) 
output_file_name = save_folder / f"{PREFIX}_{OUT_GENERATE_TIME.strftime('%Y%m%d_%H%M%S')}.csv"
output_file_name.parent.mkdir(parents=True, exist_ok=True)  # 폴더 생성

out.to_csv(output_file_name, index=False, encoding='utf-8-sig')
print(f"*** 저장 완료: {output_file_name} ({len(out):,}행) ***")


*** 저장 완료: ..\csv\문장확인\VX_오류_20260405_093745.csv (61,605행) ***


In [14]:
from pathlib import Path
import os

print("cwd:", os.getcwd())
print("resolved path:", output_file_name.resolve())
print("exists:", output_file_name.exists())

cwd: c:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\NEW_VX_script
resolved path: C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\bareun\csv\문장확인\VX_오류_2026-04-05_09:13.csv
exists: True


In [ ]:
# 문장 샘플링

import pandas as pd

group_col = "문서_었_결합_분류"   # 기준이 되는 열
n_per_group = 5                  # 각 항목마다 뽑을 문장 수
random_state = 42                # 재현 가능하게

sampled = (
    out.groupby(group_col, group_keys=False)
       .apply(lambda g: g.sample(n=min(len(g), n_per_group), random_state=random_state))
       .sort_values([group_col, "sen_id"])
       .reset_index(drop=True)
)

sampled